# 01: Exploratory Data Analysis (EDA)
## Solar Power Generation & Weather Telemetry

### Project Overview
This notebook explores the operational telemetry from two utility-scale solar photovoltaic plants located in India across 34 consecutive days (May 15 to June 17, 2020).
Each plant features:
- **22 Inverters** reporting AC and DC electrical telemetry every 15 minutes.
- **A Meteorological Weather Station** measuring Ambient Temperature (°C), PV Module Surface Temperature (°C), and Incident Global Solar Irradiation (kW/m²).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '../src')
from data_loader import load_raw_datasets, aggregate_plant_level
from data_validation import validate_raw_data, check_sampling_interval_continuity

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries successfully imported.')


### 1. Ingestion and Format-Aware Timestamp Normalization
Plant 1 generation records use `%d-%m-%Y %H:%M` while all other files use ISO `%Y-%m-%d %H:%M:%S`. The `load_raw_datasets` function standardizes all timestamps to `datetime64[ns]` and sorts chronologically.


In [ ]:
raw = load_raw_datasets('../data/raw')
validation_summary = validate_raw_data(raw)
for name, info in validation_summary.items():
    print(f"{name:10s}: {info['rows']:,} rows | Range: {info['min_timestamp']} to {info['max_timestamp']}")


### 2. Investigating Inverter Availability (Active Inverter Counts)
Each solar plant consists of 22 inverters. In Plant 2, we discover that on 840 timestamps only 18 inverters were online, representing an 18.2% capacity derating!


In [ ]:
p1_inv = raw['p1_gen'].groupby('DATE_TIME')['SOURCE_KEY'].nunique()
p2_inv = raw['p2_gen'].groupby('DATE_TIME')['SOURCE_KEY'].nunique()

print('Plant 1 Inverter Availability Breakdown:')
print(p1_inv.value_counts())
print('\nPlant 2 Inverter Availability Breakdown:')
print(p2_inv.value_counts())


### 3. Plant-Level Aggregation & Weather Join
Inverter-level AC power is summed per timestamp to produce total plant AC generation (kW), and joined with weather sensor measurements.


In [ ]:
p1_plant = aggregate_plant_level(raw['p1_gen'], raw['p1_wtr'])
p2_plant = aggregate_plant_level(raw['p2_gen'], raw['p2_wtr'])
print('Plant 1 Aggregate Shape:', p1_plant.shape)
print('Plant 2 Aggregate Shape:', p2_plant.shape)
p1_plant.head()


### 4. Correlation and Diurnal Patterns
Solar generation exhibits strong correlation with solar irradiance and panel module temperature.


In [ ]:
corr_cols = ['AC_POWER', 'DC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
corr_matrix = p1_plant[corr_cols].corr()
print('Correlation Matrix (Plant 1):')
print(corr_matrix.round(3))
